# Financial Verification with Snowflake-managed MCP

Self-contained notebook: use the active Snowflake session and warehouse. No CLI, generated folder, connector credentials or dataset download is needed. Run cells sequentially, not Run All.

This initial live deployment demonstrates synthetic distribution reconciliation for PE/fund operations. Capital-markets, hedge-fund and quant workflows can use the same verified-tool pattern but are not implemented by this one function. This is not FinanceBench scoring or a valuation engine. The final section adds a real MCP HTTP client and optional TruLens evaluations in Snowflake AI Observability; it does not add ML Experiment Tracking.

Configure an existing standard database in your account, then explicitly approve setup and MCP creation separately. No automatic grants, role changes, database creation, replacement or cleanup. DDL is not transactional; stop on errors and inspect partial state before retrying. The notebook uses `get_active_session()` (the exact Snowpark API name).

SQL checks below invoke the function directly. They are NOT MCP protocol calls. Creating a managed MCP server provides an endpoint for a separately authenticated MCP client; `get_active_session()` alone does not configure such a client. Never extract private session tokens or paste secrets into notebook cells.

In [ ]:
from snowflake.snowpark.context import get_active_session
from datetime import datetime, timezone
from decimal import Decimal
import json
import re

session = get_active_session()
EXPECTED_ACCOUNT = '<your account>'
DATABASE = 'YOUR_DATABASE'  # existing approved standard database, not USER$...
SCHEMA = '<your schema>'
WAREHOUSE = None  # None uses your current active warehouse; or enter its name
APPROVE_SETUP = False
APPROVE_CREATE_MCP = False
query_evidence = []
setup_attempted = False
checks_verified = False

def identifier(value):
    if not isinstance(value, str) or not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', value):
        raise ValueError('Use a simple unquoted identifier')
    return value.upper()

def context():
    return session.sql('SELECT CURRENT_ACCOUNT_NAME() AS ACCOUNT_NAME, CURRENT_ROLE() AS ROLE_NAME, CURRENT_DATABASE() AS DATABASE_NAME, CURRENT_WAREHOUSE() AS WAREHOUSE_NAME').collect()[0].as_dict()

def checked_context():
    actual = context()
    if actual['ACCOUNT_NAME'] != EXPECTED_ACCOUNT:
        raise RuntimeError(f"Expected {EXPECTED_ACCOUNT}; active account is {actual['ACCOUNT_NAME']}")
    if DATABASE == 'YOUR_DATABASE' or DATABASE.upper() in {'SNOWFLAKE'}:
        raise ValueError('Set DATABASE to an approved existing standard database')
    identifier(DATABASE)
    identifier(SCHEMA)
    return actual

def execute(statement, label):
    checked_context()
    event = {'label': label, 'utc': datetime.now(timezone.utc).isoformat()}
    history = None
    try:
        with session.query_history() as history:
            rows = session.sql(statement).collect()
        event.update(status='ok', rows=[row.as_dict() for row in rows])
        return rows
    except Exception as exc:
        event.update(status='error', error_type=type(exc).__name__)
        raise
    finally:
        event['query_ids'] = [query.query_id for query in history.queries] if history is not None else []
        query_evidence.append(event)

print(context())  # read-only; inspect before approving any changes

In [ ]:
actual = checked_context()
namespace = f'{identifier(DATABASE)}.{identifier(SCHEMA)}'
warehouse = identifier(WAREHOUSE or actual['WAREHOUSE_NAME'])
execute(f'USE WAREHOUSE {warehouse}', 'select_compute')
execute(f'DESCRIBE DATABASE {identifier(DATABASE)}', 'verify_database')
print({'target': namespace, 'warehouse': warehouse, 'account': actual['ACCOUNT_NAME']})

setup_statements = [
    f"CREATE SCHEMA {namespace} COMMENT = 'Isolated synthetic verification demo; no production data'",
    f'''CREATE TABLE {namespace}.WATERFALL_CASES (
        CASE_ID VARCHAR NOT NULL, PROCEEDS NUMBER(18,2) NOT NULL,
        EXPECTED_COUNT NUMBER(10,0) NOT NULL, TOLERANCE NUMBER(18,2) NOT NULL,
        CURRENCY VARCHAR NOT NULL)''',
    f'''INSERT INTO {namespace}.WATERFALL_CASES VALUES
        ('BALANCED',300000000,2,0.01,'USD'), ('SHORTFALL',300000000,2,0.01,'USD'),
        ('OVERALLOCATED',300000000,2,0.01,'USD'), ('MISSING',300000000,2,0.01,'USD'),
        ('INCOMPLETE',300000000,2,0.01,'USD')''',
    f'''CREATE TABLE {namespace}.WATERFALL_ALLOCATIONS (
        CASE_ID VARCHAR NOT NULL, ALLOCATION_ID VARCHAR NOT NULL, AMOUNT NUMBER(18,2))''',
    f'''INSERT INTO {namespace}.WATERFALL_ALLOCATIONS VALUES
        ('BALANCED','B1',200000000),('BALANCED','B2',100000000),
        ('SHORTFALL','S1',200000000),('SHORTFALL','S2',37000000),
        ('OVERALLOCATED','O1',200000000),('OVERALLOCATED','O2',101000000),
        ('INCOMPLETE','I1',200000000),('INCOMPLETE','I2',NULL)''',
    f'''CREATE VIEW {namespace}.WATERFALL_RECONCILIATION AS
    WITH allocation_summary AS (
      SELECT CASE_ID, COUNT(*) AS ACTUAL_COUNT,
        COUNT(*) - COUNT(AMOUNT) AS NULL_AMOUNT_COUNT,
        COUNT(*) - COUNT(DISTINCT ALLOCATION_ID) AS DUPLICATE_COUNT,
        SUM(IFF(AMOUNT < 0, 1, 0)) AS NEGATIVE_COUNT, SUM(AMOUNT) AS OBSERVED_TOTAL
      FROM {namespace}.WATERFALL_ALLOCATIONS GROUP BY CASE_ID
    ), checked AS (
      SELECT cases.*, COALESCE(allocations.ACTUAL_COUNT,0) AS ACTUAL_COUNT,
        COALESCE(allocations.NULL_AMOUNT_COUNT,0) AS NULL_AMOUNT_COUNT,
        COALESCE(allocations.DUPLICATE_COUNT,0) AS DUPLICATE_COUNT,
        COALESCE(allocations.NEGATIVE_COUNT,0) AS NEGATIVE_COUNT, allocations.OBSERVED_TOTAL,
        (cases.EXPECTED_COUNT > 0 AND cases.PROCEEDS >= 0 AND cases.TOLERANCE >= 0
         AND cases.CURRENCY = 'USD' AND COUNT(*) OVER (PARTITION BY cases.CASE_ID) = 1
         AND COALESCE(allocations.ACTUAL_COUNT,0) = cases.EXPECTED_COUNT
         AND COALESCE(allocations.NULL_AMOUNT_COUNT,0) = 0
         AND COALESCE(allocations.DUPLICATE_COUNT,0) = 0
         AND COALESCE(allocations.NEGATIVE_COUNT,0) = 0) AS COMPLETE
      FROM {namespace}.WATERFALL_CASES cases
      LEFT JOIN allocation_summary allocations ON cases.CASE_ID = allocations.CASE_ID
    )
    SELECT *, IFF(COMPLETE,OBSERVED_TOTAL,NULL) AS ALLOCATED_TOTAL,
      IFF(COMPLETE,PROCEEDS-OBSERVED_TOTAL,NULL) AS GAP,
      CASE WHEN NOT COMPLETE THEN 'blocked_incomplete'
           WHEN ABS(PROCEEDS-OBSERVED_TOTAL)>TOLERANCE THEN 'blocked_reconciliation'
           ELSE 'checks_passed' END AS STATUS FROM checked''',
    f'''CREATE FUNCTION {namespace}.REVIEW_RECONCILIATION(P_CASE_ID VARCHAR)
    RETURNS VARIANT LANGUAGE SQL AS $$
      SELECT IFF(COUNT(*)=1,
        ANY_VALUE(OBJECT_CONSTRUCT_KEEP_NULL(
          'case_id',results.CASE_ID,'currency',results.CURRENCY,
          'proceeds',results.PROCEEDS,'allocated_total',results.ALLOCATED_TOTAL,
          'gap',results.GAP,'tolerance',results.TOLERANCE,
          'expected_count',results.EXPECTED_COUNT,'actual_count',results.ACTUAL_COUNT,
          'null_amount_count',results.NULL_AMOUNT_COUNT,'duplicate_count',results.DUPLICATE_COUNT,
          'negative_count',results.NEGATIVE_COUNT,'status',results.STATUS,
          'source','WATERFALL_RECONCILIATION','fixture_version','v1',
          'synthetic',TRUE,'payment_authorized',FALSE)),
        OBJECT_CONSTRUCT('status',IFF(COUNT(*)=0,'blocked_unknown_case','blocked_nonunique_case'),
                         'synthetic',TRUE,'payment_authorized',FALSE))
      FROM {namespace}.WATERFALL_RECONCILIATION results WHERE results.CASE_ID=P_CASE_ID
    $$''',
]
for statement in setup_statements:
    print(statement + ';\n')

## Deploy reviewed synthetic fixtures

After reviewing the printed statements, set `APPROVE_SETUP = True` in a separate cell immediately before the next cell. Do not rerun initialization to approve: that resets evidence. This creates a **new schema**; a collision stops execution. There is no destructive reset. After partial failure or kernel restart, inspect existing objects before proceeding. The in-kernel attempt guard is not persistent.

This minimal MCP notebook intentionally omits stages, agents, scheduled monitors and managed evaluations. Warehouse compute charges apply. It retains the same five reconciliation fixtures; it does not rerun a full waterfall algorithm.

In [ ]:
if not APPROVE_SETUP:
    raise PermissionError('Review the exact SQL and set APPROVE_SETUP = True first')
APPROVE_SETUP = False
if setup_attempted:
    raise RuntimeError('Setup was already attempted; inspect state instead of rerunning')
if namespace != f'{identifier(DATABASE)}.{identifier(SCHEMA)}':
    raise RuntimeError('Configuration changed after SQL review')
setup_attempted = True
checks_verified = False
for index, statement in enumerate(setup_statements):
    execute(statement, f'setup_{index}')
print('Synthetic fixtures and read-only function created; not yet an MCP invocation.')

In [ ]:
checks_verified = False
rows = execute(f'SELECT CASE_ID, STATUS, PROCEEDS, ALLOCATED_TOTAL, GAP FROM {namespace}.WATERFALL_RECONCILIATION ORDER BY CASE_ID', 'direct_reconciliation_check')
expected = {
    'BALANCED': ('checks_passed', Decimal('0')),
    'SHORTFALL': ('blocked_reconciliation', Decimal('63000000')),
    'OVERALLOCATED': ('blocked_reconciliation', Decimal('-1000000')),
    'MISSING': ('blocked_incomplete', None),
    'INCOMPLETE': ('blocked_incomplete', None),
}
assert len(rows) == 5
assert {row['CASE_ID'] for row in rows} == set(expected)
for row in rows:
    assert (row['STATUS'], row['GAP']) == expected[row['CASE_ID']]
    if row['GAP'] is None:
        assert row['ALLOCATED_TOTAL'] is None
function_results = {}
for case in (*expected, 'UNKNOWN'):
    # case values are from the fixed local allowlist above, not user input.
    result = execute(f"SELECT {namespace}.REVIEW_RECONCILIATION('{case}') AS RESULT", f'direct_function_{case}')[0]['RESULT']
    result = json.loads(result) if isinstance(result, str) else result
    assert result['synthetic'] is True and result['payment_authorized'] is False
    assert result['status'] == (expected[case][0] if case in expected else 'blocked_unknown_case')
    function_results[case] = result
checks_verified = True
import pandas as pd
from IPython.display import display
display(pd.DataFrame([row.as_dict() for row in rows]))
print('All direct SQL assertions passed. These are not tools/call results.')

In [ ]:
mcp_name = namespace + '.FINANCIAL_VERIFICATION_MCP'
mcp_specification = f'''tools:
  - name: review_distribution_reconciliation
    title: Review synthetic distribution reconciliation
    type: GENERIC
    identifier: {namespace}.REVIEW_RECONCILIATION
    description: Review one synthetic distribution case for completeness and conservation. Returns evidence and blocking status. Does not validate recipient entitlement or authorize payments.
    config:
      type: function
      warehouse: {warehouse}
      query_timeout: 30
      input_schema:
        type: object
        properties:
          P_CASE_ID:
            type: string
            description: Synthetic case identifier; unknown cases are blocked.
        required: [P_CASE_ID]
        additionalProperties: false
'''
create_mcp_sql = f'CREATE MCP SERVER {mcp_name} FROM SPECIFICATION $$\n{mcp_specification}$$'
print(create_mcp_sql)
print('Review, then set APPROVE_CREATE_MCP = True in a separate cell before deployment.')

In [ ]:
if not APPROVE_CREATE_MCP:
    raise PermissionError('Approve the reviewed managed MCP specification first')
APPROVE_CREATE_MCP = False
if not checks_verified:
    raise RuntimeError('Run and pass the direct verification checks first')
if namespace != f'{identifier(DATABASE)}.{identifier(SCHEMA)}':
    raise RuntimeError('Configuration changed after review')
execute(create_mcp_sql, 'create_managed_mcp')
metadata = execute(f'DESCRIBE MCP SERVER {mcp_name}', 'inspect_managed_mcp')
display(pd.DataFrame([row.as_dict() for row in metadata]))
print('Managed MCP server created. No MCP client has been authenticated or invoked by this cell.')

## Capture real MCP calls separately

Connect an approved MCP client with OAuth to:

`https://<account-host>/api/v2/databases/<DATABASE>/schemas/<SCHEMA>/mcp-servers/FINANCIAL_VERIFICATION_MCP`

Use the account's verified hostname (hyphens, not underscores). The client role needs access to the MCP server and underlying function/warehouse; this notebook does not grant it. No external MCP hosting is needed. A client must support the documented Snowflake MCP response transport, including `application/json` and `text/event-stream`.

Capture `tools/list`, then `tools/call` for `review_distribution_reconciliation` with `{"P_CASE_ID":"SHORTFALL"}`, plus BALANCED and UNKNOWN. Preserve actual responses, errors and request identifiers. Expected SHORTFALL evidence is a 63000000 gap and blocked reconciliation; no result authorizes payment. Do not use direct SQL output as protocol evidence.

The client and TruLens evaluation cells below provide the authenticated HTTP path. They require an OAuth token supplied through an approved runtime secret/environment mechanism and network access to the account host. The active Snowpark session is used only for setup and evaluation registration, not as a source of private authentication tokens. FinanceBench ingestion, turbopuffer retrieval and ML Experiment Tracking remain separate integrations.

Reference: https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-mcp

In [ ]:
# Explicit local runtime export; download/persist these files before shutdown.
# This contains SQL execution evidence, NOT MCP protocol responses.
from pathlib import Path

def export_sql_evidence():
    output = Path('evidence') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output.mkdir(parents=True, exist_ok=False)
    (output / 'sql_execution_evidence.json').write_text(json.dumps(query_evidence, indent=2, default=str))
    (output / 'managed_mcp.yaml').write_text(mcp_specification)
    print(f'Saved to {output}; direct SQL evidence only. Review before sharing.')

print('Call export_sql_evidence() after execution to retain query IDs and results.')

## MCP client and Snowflake-hosted evaluations

This section evaluates a deterministic MCP client wrapper, not an autonomous LLM tool-selector. Each input invokes the live reconciliation tool; output is the returned evidence, not a recomputed SQL result. TruLens registers this custom workflow as an **External Agent** and stores traces/evaluation results in Snowflake AI Observability. Native `EXECUTE_AI_EVALUATION` is not used.

Prerequisites: compatible `trulens-core`, `trulens-connectors-snowflake`, `trulens-providers-cortex` (Snowflake documents 2.1.2 or later), and `requests` in the notebook runtime. No packages are installed automatically. Review and pin your environment before the run; installed versions are recorded below. Your role needs the documented External Agent/task/Cortex evaluation privileges. No grants are applied here.

Configure notebook network access to your Snowflake account hostname and provide a short-lived OAuth access token through the approved runtime environment as `MCP_OAUTH_TOKEN`. This notebook does not perform interactive OAuth or renew tokens. Never put a token in a cell, export, prompt or screenshot. Set `MCP_ACCOUNT_HOST` to your verified account hostname; confirm the OAuth identity/role belongs to the intended account. No private Snowpark token is read. If network access or token injection is unavailable in your runtime, stop and configure it rather than substituting direct SQL.

Approvals are separate: one discovery call; registration plus a six-input invocation run; then server-side judging. Warehouse, task, trace-storage and judge charges can apply. There is no hard credit cap. Failed requests remain evidence and are never silently retried. Do not rerun initialization midway through a run.

Sources: https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/evaluate-applications-trulens and https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference

In [ ]:
import os
import time
import uuid
import threading
import hashlib
from importlib.metadata import version
from urllib.parse import urlsplit
import requests

MCP_ACCOUNT_HOST = ''  # verified hostname, e.g. org-account.snowflakecomputing.com
APPROVE_MCP_DISCOVERY = False
APPROVE_TRULENS_RUN = False
APPROVE_JUDGE_METRICS = False
JUDGE_MODEL = ''  # explicitly choose an allowed Cortex judge model
mcp_events = []
eval_attempted = False
metrics_attempted = False


def parse_mcp_response(body, media_type, request_id):
    text = body.decode('utf-8')
    if media_type.split(';')[0].strip() == 'text/event-stream':
        messages = []
        for block in re.split(r'\r?\n\r?\n', text):
            data = '\n'.join(line[5:].lstrip(' ') for line in block.splitlines() if line.startswith('data:'))
            if data and data != '[DONE]':
                messages.append(json.loads(data))
    elif media_type.split(';')[0].strip() == 'application/json':
        messages = [json.loads(text)]
    else:
        raise ValueError('Unexpected MCP response content type')
    matches = [item for item in messages if isinstance(item, dict) and item.get('id') == request_id]
    if len(matches) != 1 or matches[0].get('jsonrpc') != '2.0':
        raise ValueError('Missing or ambiguous MCP response ID')
    message = matches[0]
    if 'error' in message or not isinstance(message.get('result'), dict):
        raise ValueError('MCP JSON-RPC error or missing result')
    return message


class SnowflakeMCPHTTP:
    def __init__(self, host, database, schema, max_calls=7):
        if not re.fullmatch(r'[a-z0-9][a-z0-9.-]*\.snowflakecomputing\.com', host):
            raise ValueError('Set a verified HTTPS Snowflake hostname without underscores')
        self.endpoint = f'https://{host}/api/v2/databases/{identifier(database)}/schemas/{identifier(schema)}/mcp-servers/FINANCIAL_VERIFICATION_MCP'
        self.max_calls = max_calls
        self.calls = 0
        self.lock = threading.Lock()
        self.discovered = False

    def request(self, method, params=None):
        with self.lock:
            if self.calls >= self.max_calls:
                raise RuntimeError('MCP request budget exhausted; no automatic reruns')
            self.calls += 1
        request_id = str(uuid.uuid4())
        payload = {'jsonrpc': '2.0', 'id': request_id, 'method': method}
        if params is not None:
            payload['params'] = params
        event = {'request': payload, 'endpoint': self.endpoint,
                 'utc': datetime.now(timezone.utc).isoformat(), 'execution_kind': 'live_mcp_http'}
        started = time.monotonic()
        try:
            token = os.environ.get('MCP_OAUTH_TOKEN', '')
            if not token or '\n' in token or '\r' in token:
                raise ValueError('Configure a valid OAuth token through the approved runtime secret mechanism')
            # Dedicated session: no netrc credentials, environment proxy or redirects.
            with requests.Session() as http:
                http.trust_env = False
                with http.post(self.endpoint, json=payload,
                               headers={'Authorization': 'Bearer ' + token,
                                        'Accept': 'application/json, text/event-stream'},
                               timeout=(10, 60), allow_redirects=False, stream=True) as response:
                    event['http_status'] = response.status_code
                    if response.status_code != 200:
                        raise RuntimeError('MCP HTTP request rejected')
                    body = bytearray()
                    for chunk in response.iter_content(chunk_size=8192):
                        body.extend(chunk)
                        if len(body) > 1024 * 1024 or time.monotonic() - started > 90:
                            raise RuntimeError('MCP response exceeded size/time budget')
                    message = parse_mcp_response(bytes(body), response.headers.get('Content-Type', ''), request_id)
            event['response'] = message
            if message['result'].get('isError'):
                raise RuntimeError('MCP tool returned an error')
            event['status'] = 'ok'
            return message['result']
        except Exception as exc:
            event.update(status='error', error_type=type(exc).__name__)
            raise RuntimeError('MCP failed; inspect sanitized mcp_events and client configuration') from None
        finally:
            event['latency_ms'] = (time.monotonic() - started) * 1000
            with self.lock:
                mcp_events.append(event)

    def discover(self):
        self.discovered = False
        result = self.request('tools/list')
        tools = result.get('tools', [])
        if result.get('nextCursor') or [tool.get('name') for tool in tools] != ['review_distribution_reconciliation']:
            raise ValueError('Review unexpected/paginated tool inventory; no tool calls enabled')
        schema = tools[0].get('inputSchema', {})
        if (schema.get('type') != 'object' or set(schema.get('properties', {})) != {'P_CASE_ID'}
                or schema['properties']['P_CASE_ID'].get('type') != 'string'
                or schema.get('required') != ['P_CASE_ID']):
            raise ValueError('Unexpected tool input schema')
        self.discovered = True
        return result

    def review(self, case_id):
        if not self.discovered:
            raise RuntimeError('Verify live tool discovery first')
        if case_id not in {'BALANCED', 'SHORTFALL', 'OVERALLOCATED', 'MISSING', 'INCOMPLETE', 'UNKNOWN'}:
            raise ValueError('Only synthetic cases are permitted')
        result = self.request('tools/call', {'name': 'review_distribution_reconciliation', 'arguments': {'P_CASE_ID': case_id}})
        # Preserve wire result as evidence; do not invent or silently unwrap unfamiliar schemas.
        return json.dumps(result, sort_keys=True)

print('Client defined; no connection or MCP call made.')

In [ ]:
if not APPROVE_MCP_DISCOVERY:
    raise PermissionError('Review account host/auth/network configuration, then approve one discovery call')
APPROVE_MCP_DISCOVERY = False
checked_context()
if 'mcp_client' in globals():
    raise RuntimeError('Client already exists; inspect its budget/events rather than resetting it')
mcp_client = SnowflakeMCPHTTP(MCP_ACCOUNT_HOST, DATABASE, SCHEMA, max_calls=7)
inventory = mcp_client.discover()
display(inventory)
print('Live tool discovery complete. Six tool calls remain in the client budget.')

In [ ]:
# Imports and instrumented class definitions only; registration is in the next gated cell.
os.environ['TRULENS_OTEL_TRACING'] = '1'
from trulens.apps.app import TruApp
from trulens.connectors.snowflake import SnowflakeConnector
from trulens.core import TruSession
from trulens.core.run import RunConfig
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

package_versions = {name: version(name) for name in
                    ('trulens-core', 'trulens-connectors-snowflake', 'trulens-providers-cortex', 'requests')}
print(package_versions)

class FinancialMCPApplication:
    @instrument(span_type=SpanAttributes.SpanType.RETRIEVAL,
                attributes={SpanAttributes.RETRIEVAL.QUERY_TEXT: 'query',
                            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: 'return'})
    def retrieve(self, query: str) -> list:
        # Fixed routing demonstrates protocol/tool correctness, not LLM tool selection.
        if query not in {'BALANCED', 'SHORTFALL', 'OVERALLOCATED', 'MISSING', 'INCOMPLETE', 'UNKNOWN'}:
            raise ValueError('Unknown evaluation input')
        return [mcp_client.review(query)]

    @instrument(span_type=SpanAttributes.SpanType.RECORD_ROOT,
                attributes={SpanAttributes.RECORD_ROOT.INPUT: 'query',
                            SpanAttributes.RECORD_ROOT.OUTPUT: 'return'})
    def respond(self, query: str) -> str:
        return self.retrieve(query)[0]

# Independently authored ground truth; never pass it to the MCP tool.
evaluation_df = pd.DataFrame([
    {'query': 'BALANCED', 'gold': 'Synthetic USD proceeds 300000000, allocated_total 300000000, gap 0, status checks_passed, payment_authorized false.'},
    {'query': 'SHORTFALL', 'gold': 'Synthetic USD proceeds 300000000, allocated_total 237000000, gap 63000000, status blocked_reconciliation, payment_authorized false.'},
    {'query': 'OVERALLOCATED', 'gold': 'Synthetic USD proceeds 300000000, allocated_total 301000000, gap -1000000, status blocked_reconciliation, payment_authorized false.'},
    {'query': 'MISSING', 'gold': 'Synthetic missing allocations, actual_count 0, expected_count 2, allocated_total null, gap null, status blocked_incomplete, payment_authorized false.'},
    {'query': 'INCOMPLETE', 'gold': 'Synthetic actual_count 2, null_amount_count 1, allocated_total null, gap null, status blocked_incomplete, payment_authorized false.'},
    {'query': 'UNKNOWN', 'gold': 'Status blocked_unknown_case, synthetic true, payment_authorized false.'},
])
dataset_hash = hashlib.sha256(evaluation_df.to_json(orient='records').encode()).hexdigest()
display(evaluation_df)
print('Six live MCP calls; correctness and groundedness judging are separately gated. No responder LLM is used.')

In [ ]:
if not APPROVE_TRULENS_RUN:
    raise PermissionError('Approve External Agent/run registration, six MCP tool calls and trace ingestion first')
APPROVE_TRULENS_RUN = False
checked_context()
if eval_attempted:
    raise RuntimeError('Run already attempted; inspect existing run and errors instead of repeating it')
if not JUDGE_MODEL.strip():
    raise ValueError('Select an allowed Cortex judge model explicitly')
if not mcp_client.discovered or mcp_client.calls != 1:
    raise RuntimeError('Expected one discovery and no prior tool calls')
if len(evaluation_df) != 6 or hashlib.sha256(evaluation_df.to_json(orient='records').encode()).hexdigest() != dataset_hash:
    raise ValueError('Evaluation dataset changed after review')
eval_attempted = True
execute(f'USE DATABASE {identifier(DATABASE)}', 'eval_database')
execute(f'USE SCHEMA {namespace}', 'eval_schema')
execute(f'USE WAREHOUSE {warehouse}', 'eval_warehouse')
connector = SnowflakeConnector(snowpark_session=session)
tru_session = TruSession(connector=connector)
application = FinancialMCPApplication()
APP_NAME = 'FINANCIAL_MCP_CLIENT_EVAL'
APP_VERSION = 'mcp_http_v1'
RUN_NAME = 'MCP_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
tru_app = TruApp(application, app_name=APP_NAME, app_version=APP_VERSION,
                 connector=connector, main_method=application.respond)
run = tru_app.add_run(run_config=RunConfig(
    run_name=RUN_NAME, description='Six synthetic cases through live Snowflake-managed MCP; dataset sha256 ' + dataset_hash,
    label='managed-mcp-synthetic-v1', source_type='DATAFRAME', dataset_name='synthetic_reconciliation_v1',
    dataset_spec={'RECORD_ROOT.INPUT': 'query', 'RETRIEVAL.QUERY_TEXT': 'query',
                  'RECORD_ROOT.GROUND_TRUTH_OUTPUT': 'gold'}, llm_judge_name=JUDGE_MODEL))
print({'app': APP_NAME, 'version': APP_VERSION, 'run': RUN_NAME, 'dataset_hash': dataset_hash})
try:
    run.start(input_df=evaluation_df)
finally:
    display(pd.DataFrame([{key: event.get(key) for key in ('utc', 'status', 'http_status', 'latency_ms', 'error_type')}
                          for event in mcp_events]))
print(run.get_status())

In [ ]:
# Read status; rerun this cell to refresh. Never rerun run.start() to poll.
print(run.get_status())
display(run.describe())

# Define only; call submit_judge_metrics() after approving billable server-side judging.
def submit_judge_metrics():
    global APPROVE_JUDGE_METRICS, metrics_attempted
    if not APPROVE_JUDGE_METRICS:
        raise PermissionError('Approve Cortex judge costs before computing metrics')
    APPROVE_JUDGE_METRICS = False
    checked_context()
    status = run.get_status()
    status_name = getattr(status, 'name', str(status).split('.')[-1])
    if status_name not in {'INVOCATION_COMPLETED', 'INVOCATION_PARTIALLY_COMPLETED'}:
        raise RuntimeError(f'Invocation not ready: {status_name}')
    if metrics_attempted:
        raise RuntimeError('Metrics already attempted; inspect status before taking further action')
    metrics_attempted = True
    return run.compute_metrics(metrics=['correctness', 'groundedness'])

print('Review traces first. Then set APPROVE_JUDGE_METRICS=True and call submit_judge_metrics().')

## Review managed evaluation evidence

Open **Snowsight → AI & ML → Evaluations → FINANCIAL_MCP_CLIENT_EVAL**, select the printed version/run, and inspect each record's retrieval span and root output. Refresh `run.get_status()` until scoring completes; retain partial failures. `run.cancel()` can cancel an active run; do not delete the run to hide failures.

This is a TruLens External Agent evaluation whose invocation runs the notebook's MCP client and whose named metrics are computed by Snowflake. It is **not** a native Cortex Agent `EXECUTE_AI_EVALUATION` run, and not ML Experiment Tracking. Direct inbound managed MCP invocations are not automatically recorded in AI_OBSERVABILITY_EVENTS; the client instrumentation supplies these spans.

Groundedness here measures fidelity to the returned tool evidence (the wrapper returns it unchanged), so a high score is expected and does not prove financial correctness. Correctness compares independent fixture expectations; neither metric validates production economics. Inspect exact monetary amounts, NULLs, statuses and payment_authorized=false as well. The model does not choose the tool or generate the answer in this minimal test.

No live calls were performed while authoring. Package compatibility, runtime networking, OAuth and target-account privileges must be verified on your run. The seven-request client budget includes one discovery and six tool calls; no automatic retry or token refresh occurs. After kernel restart, inspect saved run state before approving another execution.

In [ ]:
# Explicit export only; protocol responses contain synthetic tool data, not auth headers.
def export_mcp_evaluation_evidence():
    output = Path('evidence') / ('mcp_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    output.mkdir(parents=True, exist_ok=False)
    (output / 'mcp_protocol.json').write_text(json.dumps(mcp_events, indent=2))
    metadata = {'app': APP_NAME, 'version': APP_VERSION, 'run': RUN_NAME,
                'dataset_sha256': dataset_hash, 'packages': package_versions,
                'judge': JUDGE_MODEL, 'status': str(run.get_status()), 'endpoint': mcp_client.endpoint}
    (output / 'evaluation_metadata.json').write_text(json.dumps(metadata, indent=2))
    (output / 'run_description.json').write_text(json.dumps(run.describe(), indent=2, default=str))
    print(f'Saved {output}. Download before runtime termination; review before publication.')

print('Call export_mcp_evaluation_evidence() to retain actual protocol and evaluation metadata.')